In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jayjoshi37/inventory-demand-forecasting-and-stockout-risk")

print("Path to dataset files:", path)

step-1 need to check file name & load csv

In [ ]:
import os
import pandas as pd

# Path set karein jo kagglehub se mila
dataset_path = path

# Folder ke andar files check karein
files = os.listdir(dataset_path)
print("Files in dataset folder:", files)

# CSV file ko find karke load karein
csv_file = [f for f in files if f.endswith('.csv')][0]
full_csv_path = os.path.join(dataset_path, csv_file)

# Read Data
df = pd.read_csv(full_csv_path)

# Data ka pehla look check karein
print("\n--- Data Head ---")
display(df.head())

print("\n--- Data Info ---")
print(df.info())

Step 2: Exploratory Data Analysis (EDA) & Cleaning

In [ ]:
# Missing values check karein
print("Missing Values:\n", df.isnull().sum())

# Data Summary Statistics
display(df.describe())

# Columns ke naam check karein
print("\nColumns in Dataset:", df.columns.tolist())

In [ ]:
import os
import pandas as pd
from google.colab import files

# 1. Dataset folder se CSV file ka path nikalein
files_in_dir = os.listdir(path)
csv_filename = [f for f in files_in_dir if f.endswith('.csv')][0]
full_csv_path = os.path.join(path, csv_filename)

# 2. Data read karein
df = pd.read_csv(full_csv_path)

# 3. Cleaned CSV ko Colab se aapke System/Laptop par download karein
df.to_csv("inventory_data.csv", index=False)
files.download("inventory_data.csv")

print("File Download Success! Total Rows:", len(df))
display(df.head())

In [ ]:
# Model Preparation Code
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Preprocessing
df_clean = df.dropna()
X = df_clean.drop(columns=['stockout_risk']) if 'stockout_risk' in df_clean.columns else df_clean.iloc[:, :-1]
y = df_clean['stockout_risk'] if 'stockout_risk' in df_clean.columns else df_clean.iloc[:, -1]

# Categorical data handling
X = pd.get_dummies(X, drop_first=True)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print("--- Model Accuracy & Metrics ---")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Copy data
df_model = df.copy()

# Non-useful columns drop karein (agar hain)
for col in ['product_id', 'date', 'store_id']:
    if col in df_model.columns:
        df_model = df_model.drop(columns=[col])

# Categorical text variables ko numerical mein convert karein
df_encoded = pd.get_dummies(df_model, drop_first=True)

# Target variable identify karein
target_col = [c for c in df_encoded.columns if 'stockout' in c.lower()][0]
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

# Train-Test Split with Stratify
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Class Weight Balanced Model (Fixes 0.00 score issue)
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Output evaluation
y_pred = model.predict(X_test)
print("--- Final Model Performance ---")
print(classification_report(y_test, y_pred))

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Handle Class Imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 2. Train Model on Balanced Data
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train_res, y_train_res)

# 3. Final Prediction
y_pred_final = final_model.predict(X_test)

print("--- Improved Model Performance (Balanced) ---")
print(classification_report(y_test, y_pred_final))

In [ ]:
import shutil
import os
import kagglehub

# Ensure the dataset path is valid by re-downloading or getting the existing path
path = kagglehub.dataset_download("jayjoshi37/inventory-demand-forecasting-and-stockout-risk")

# Re-defining full_csv_path to ensure it's available in this cell's scope
files_in_dir = os.listdir(path)
csv_filename = [f for f in files_in_dir if f.endswith('.csv')][0]
full_csv_path = os.path.join(path, csv_filename)

shutil.copy(full_csv_path, "./inventory_data.csv")
print("File successfully copied to visual folder!")